In [21]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import textwrap

# =====================================================
# HELPER: WRAP LABELS
# =====================================================
def wrap_labels(labels, width=10):
    return ['\n'.join(textwrap.wrap(str(label), width)) for label in labels]

# =====================================================
# PATHS
# =====================================================

BOOK_PATH = "/home/moshtasa/Research/phd-svd-recsys/Book/data/genre_clean.csv"
MOVIE_PATH = "/home/moshtasa/Research/phd-svd-recsys/Movie /data/df_final.csv"
MOBILE_PATH = "/home/moshtasa/Research/phd-svd-recsys/Mobile/0316_Single/data/data/mobilerec_final.csv"

OUT_DIR = Path("/home/moshtasa/Research/phd-svd-recsys/state of the art/paper_figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# ================== BOOK DATA =========================
# =====================================================

df_books = pd.read_csv(BOOK_PATH)

genres_split = df_books["genres"].astype(str).str.split(",", n=1, expand=True)
df_books["g1"] = genres_split[0].str.strip()
df_books["g2"] = genres_split[1].str.strip() if 1 in genres_split.columns else ""

books = df_books[["book_id", "g1", "g2"]].drop_duplicates()

first_counts = books["g1"].value_counts()
second_counts = books["g2"].value_counts()

all_genres = sorted(set(first_counts.index).union(set(second_counts.index)))

summary_books = pd.DataFrame({
    "genre": all_genres,
    "total": (first_counts.reindex(all_genres, fill_value=0) +
              second_counts.reindex(all_genres, fill_value=0)).values
}).sort_values("total", ascending=False)

# =====================================================
# ================== MOVIE DATA ========================
# =====================================================

df_movies = pd.read_csv(MOVIE_PATH)

df_movies["decade"] = df_movies["decade"].fillna(-1).astype(int).astype(str)
df_movies = df_movies[df_movies["decade"] != "-1"]

movies = df_movies[["item_id", "decade"]].drop_duplicates()

summary_movies = movies["decade"].value_counts().reset_index()
summary_movies.columns = ["decade", "total"]
summary_movies = summary_movies.sort_values("total", ascending=False)

# =====================================================
# ================== MOBILE DATA =======================
# =====================================================

df_mobile = pd.read_csv(MOBILE_PATH, low_memory=False)

category_mapping = {
"Action":"Games","Puzzle":"Games","Simulation":"Games","Strategy":"Games","Casual":"Games",
"Adventure":"Games","Sports":"Games","Arcade":"Games","Casino":"Games","Card":"Games",
"Board":"Games","Word":"Games","Racing":"Games","Role Playing":"Games","Trivia":"Games",
"Education":"Education & Learning","Educational":"Education & Learning",
"Books & Reference":"Education & Learning","Libraries & Demo":"Education & Learning",
"Health & Fitness":"Health & Lifestyle","Lifestyle":"Health & Lifestyle",
"Beauty":"Health & Lifestyle","Parenting":"Health & Lifestyle",
"Entertainment":"Media & Entertainment","Music":"Media & Entertainment",
"Music & Audio":"Media & Entertainment","Video Players & Editors":"Media & Entertainment",
"Comics":"Media & Entertainment",
"Social":"Social & Communication","Communication":"Social & Communication",
"Dating":"Social & Communication",
"Shopping":"Shopping & Food","Food & Drink":"Shopping & Food",
"Travel & Local":"Travel & Navigation","Maps & Navigation":"Travel & Navigation",
"Auto & Vehicles":"Travel & Navigation",
"Productivity":"Work & Productivity","Business":"Work & Productivity",
"Finance":"Work & Productivity","Tools":"Work & Productivity",
"Weather":"Utilities & Info","News & Magazines":"Utilities & Info",
"Photography":"Utilities & Info","Art & Design":"Utilities & Info",
"Personalization":"Utilities & Info","House & Home":"Utilities & Info",
"Events":"Utilities & Info"
}

df_mobile["category_mapped"] = df_mobile["app_category"].map(category_mapping)

mapped_counts = (
    df_mobile[["app_package","category_mapped"]]
    .drop_duplicates()["category_mapped"]
    .value_counts()
    .sort_values(ascending=False)
)

# =====================================================
# STYLE
# =====================================================

def apply_style(ax):
    ax.yaxis.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
        spine.set_color("black")

# =====================================================
# FIGURE 1: BOOKS
# =====================================================

fig, ax = plt.subplots(figsize=(10, 6))  # wider

x = np.arange(len(summary_books))

ax.bar(x, summary_books["total"],
       width=0.6, color="#df80ad", edgecolor="#c6186b")

ax.set_title("Books by Genre", fontsize=12, fontweight='bold')
ax.set_xlabel("Genre", fontsize=12)
ax.set_ylabel("Number of Unique Books", fontsize=11)

ax.set_xticks(x)
ax.set_xticklabels(wrap_labels(summary_books["genre"], 10), rotation=0, fontsize=9)

apply_style(ax)

plt.tight_layout()
plt.savefig(OUT_DIR / "books_genre.png", dpi=600)
plt.close()

# =====================================================
# FIGURE 2: MOVIES
# =====================================================

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(summary_movies))

ax.bar(x, summary_movies["total"],
       width=0.6, color="#a1adf2", edgecolor="#2e2da8")

ax.set_title("Movies by Decade", fontsize=12, fontweight='bold')
ax.set_xlabel("Decade", fontsize=12)
ax.set_ylabel("Number of Unique Movies", fontsize=11)

ax.set_xticks(x)
ax.set_xticklabels(summary_movies["decade"], rotation=0, fontsize=10)

apply_style(ax)

plt.tight_layout()
plt.savefig(OUT_DIR / "movies_decade.png", dpi=600)
plt.close()

# =====================================================
# FIGURE 3: MOBILE
# =====================================================

fig, ax = plt.subplots(figsize=(11, 6))

x = np.arange(len(mapped_counts))

ax.bar(x, mapped_counts.values,
       width=0.6, color="#afcb83", edgecolor="#507d09")

ax.set_title("Mobile Apps by Category", fontsize=13, fontweight='bold')
ax.set_xlabel("Category", fontsize=12)
ax.set_ylabel("Number of Unique Applications", fontsize=12)

ax.set_xticks(x)
ax.set_xticklabels(wrap_labels(mapped_counts.index, 12), rotation=0, fontsize=9)

apply_style(ax)

plt.tight_layout()
plt.savefig(OUT_DIR / "mobile_domains.png", dpi=600)
plt.close()

print("Saved all three figures to:")
print(OUT_DIR)